In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
df = pd.read_csv("../data/loan_history.csv")
df.head()

In [ ]:
# Check DataFrame structure and summary
df.info()

# Check for missing values
missing_values = df.isnull().sum()
print("Missing Values:\n", missing_values)

# Check for duplicate rows & remove them
duplicate_count = df.duplicated().sum()
print("Number of Duplicate Rows:", duplicate_count)
df = df.drop_duplicates()

In [ ]:
# Convert date columns to datetime format
df['DisbursementDate'] = pd.to_datetime(df['DisbursementDate'])
df['DueDate'] = pd.to_datetime(df['DueDate'])
df['ActualRepaymentDate'] = pd.to_datetime(df['ActualRepaymentDate'])

# Ensure PastDue30Days is numeric
df['PastDue30Days'] = pd.to_numeric(df['PastDue30Days'], errors='coerce')

print("\nData Types After Conversion:")
print(df.dtypes)

In [ ]:
# Create a new column to indicate late payments
df['LatePayment'] = df['PastDue30Days'] == 1  # True if repayment is overdue by 30 days or more
print("\nUpdated DataFrame with Late Payment Status:")
print(df[['CustomerID', 'PastDue30Days', 'LatePayment']].head())

In [ ]:
# Sort the data chronologically per customer
df = df.sort_values(by=['CustomerID', 'DisbursementDate'])

# Count how many previous loans each customer has had
df['previous_loans'] = df.groupby('CustomerID').cumcount()

# Flag if current loan was on time
df['on_time'] = df['PastDue30Days'] == 0

# Calculate how many on-time repayments the customer had before this loan
df['previous_on_time'] = df.groupby('CustomerID')['on_time'].cumsum() - df['on_time']

# Check if the customer has ever had a loan 30+ days past due up to this point
df['ever_30dpd'] = df.groupby('CustomerID')['LatePayment'].cummax()

df.head()

In [ ]:
# Save the cleaned and enriched loan history data
df.to_csv("../data/clean_loan_history.csv", index=False)
# Reset index for further analysis
df = df.sort_values(['CustomerID', 'DisbursementDate']).reset_index(drop=True)

In [ ]:
# Load simulation result file (assuming simulation_engine.py has been run)
df_sim = pd.read_csv("../data/simulation_results.csv")
df_sim.head()

In [ ]:
# Add Rule-Based Approval
df_sim["IsApproved_By_Rule"] = df_sim["LoanAmount"] <= df_sim["AllowedLimit"]

# Define Actual Risk (mapped from 'ever_30dpd' in cleaning step or present in sim results)
df_sim["Actual_Bad"] = df_sim["ever_30dpd"] == True

# Classification logic
def classify(row):
    if not row["IsApproved_By_Rule"] and row["Actual_Bad"]:
        return "True Positive"
    elif not row["IsApproved_By_Rule"] and not row["Actual_Bad"]:
        return "False Positive"
    elif row["IsApproved_By_Rule"] and row["Actual_Bad"]:
        return "False Negative"
    else:
        return "True Negative"

df_sim["Classification"] = df_sim.apply(classify, axis=1)
df_sim["Classification"].value_counts()

In [ ]:
# Calculate Mitigated Risk
mitigated_risk = df_sim[df_sim["Classification"] == "True Positive"]["LoanAmount"].sum()
print("Mitigated Risk:", mitigated_risk)

In [ ]:
# Visualization of Loan Amount Trend
df_sim["DisbursementDate"] = pd.to_datetime(df_sim["DisbursementDate"])
df_sim = df_sim.sort_values("DisbursementDate")

plt.figure(figsize=(10, 6))
plt.plot(df_sim["DisbursementDate"], df_sim["LoanAmount"], marker='o')
plt.xticks(rotation=45)
plt.title("Loan Trend")
plt.xlabel("Disbursement Date")
plt.ylabel("Loan Amount")
plt.grid(True)
plt.show()